# Day 7 Lab 04: Bronze Raw Ingestion

        **Layer:** Bronze  
        **Python reference:** `Lab_Files/labs/lab_04_bronze_raw_ingestion.py`

        This notebook is a sectioned companion version of the existing Python lab. It does not replace `run_lab.py` or the scripts under `Lab_Files/labs`.

        ## Learning Checkpoints
        - Read all raw order-event batches.
- Add Bronze metadata columns.
- Write the raw Bronze Parquet table and inspect counts by source file.

## 0. Notebook Setup

Run this first. It moves the kernel into `Lab_Files` and makes the shared lab helpers importable.

In [1]:
from pathlib import Path
import os
import sys
import types
import typing

# PySpark 3.4 imports typing.io, which is absent in Python 3.14+.
if "typing.io" not in sys.modules:
    typing_io = types.ModuleType("typing.io")
    typing_io.IO = typing.IO
    typing_io.TextIO = typing.TextIO
    typing_io.BinaryIO = typing.BinaryIO
    sys.modules["typing.io"] = typing_io

def find_lab_files(start: Path) -> Path:
    relative_options = [
        Path("."),
        Path("Lab_Files"),
        Path("Day_07") / "Lab_Files",
        Path("Week_02") / "Day_07" / "Lab_Files",
    ]
    for root in [start, *start.parents]:
        for relative in relative_options:
            candidate = (root / relative).resolve()
            if (candidate / "labs" / "day7_common.py").exists():
                return candidate
    raise FileNotFoundError(
        "Could not find Week_02/Day_07/Lab_Files. "
        "Start Jupyter from the repository root, Week_02/Day_07, or Week_02/Day_07/Lab_Files."
    )

HERE = Path.cwd().resolve()
LAB_FILES = find_lab_files(HERE)

os.chdir(LAB_FILES)
labs_path = str(LAB_FILES / "labs")
if labs_path not in sys.path:
    sys.path.insert(0, labs_path)

print(f"Working directory: {Path.cwd()}")
print(f"Using lab helpers from: {labs_path}")


Working directory: C:\Users\Gamer\Documents\GitHub\Medallion pipeline\Week_02\Day_07\Lab_Files
Using lab helpers from: C:\Users\Gamer\Documents\GitHub\Medallion pipeline\Week_02\Day_07\Lab_Files\labs


## 1. Import Lab Helpers

In [2]:

import importlib
import day7_common
day7_common = importlib.reload(day7_common)

from day7_common import LAKE_DIR, OUTPUT_DIR, clean_lab_dir, ensure_output_dirs, read_order_events, require_source_data, spark_session, with_bronze_metadata, write_csv_dir, read_parquet, write_parquet

## 2. Start Spark

In [3]:
require_source_data()
ensure_output_dirs()
spark = spark_session("Day7Notebook04BronzeRawIngestion")

## 3. Read Raw Order Events

In [4]:
raw_orders = read_order_events(spark)
raw_orders.printSchema()
raw_orders.show(10, truncate=False)

root
 |-- event_id: string (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- status: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- coupon_code: string (nullable = true)
 |-- delivery_promise: string (nullable = true)

+--------+--------+-----------+----------+---------+-------+--------+-------+--------------------+-----------+----------------+
|event_id|order_id|customer_id|product_id|status   |amount |currency|channel|event_time          |coupon_code|delivery_promise|
+--------+--------+-----------+----------+---------+-------+--------+-------+--------------------+-----------+----------------+
|evt-1007|1001    |501        |P-LAP-01  |SHIPPED  |1299.99|USD     |web    |2026-05-30T09:00:01Z|SPRING10   |2-day           |
|evt-1008|1006    |504     

## 4. Add Bronze Metadata

In [5]:
bronze = with_bronze_metadata(raw_orders, "notebook-full-load-001")
bronze.select("event_id", "order_id", "_source_file", "_ingestion_batch_id", "_bronze_ingested_at").show(10, truncate=False)

+--------+--------+--------------------------+----------------------+--------------------------+
|event_id|order_id|_source_file              |_ingestion_batch_id   |_bronze_ingested_at       |
+--------+--------+--------------------------+----------------------+--------------------------+
|evt-1007|1001    |order_events_batch_2.jsonl|notebook-full-load-001|2026-06-09 07:31:20.860729|
|evt-1008|1006    |order_events_batch_2.jsonl|notebook-full-load-001|2026-06-09 07:31:20.860729|
|evt-1009|1007    |order_events_batch_2.jsonl|notebook-full-load-001|2026-06-09 07:31:20.860729|
|evt-1010|1008    |order_events_batch_2.jsonl|notebook-full-load-001|2026-06-09 07:31:20.860729|
|evt-1011|1002    |order_events_batch_2.jsonl|notebook-full-load-001|2026-06-09 07:31:20.860729|
|evt-1012|1009    |order_events_batch_2.jsonl|notebook-full-load-001|2026-06-09 07:31:20.860729|
|evt-1001|1001    |order_events_batch_1.jsonl|notebook-full-load-001|2026-06-09 07:31:20.860729|
|evt-1002|1002    |order_event

## 5. Write Bronze

In [6]:
bronze_path = clean_lab_dir(LAKE_DIR / "bronze" / "orders_raw")
write_parquet(bronze, bronze_path, mode="overwrite")
print(f"Bronze path: {bronze_path}")

Bronze path: C:\Users\Gamer\Documents\GitHub\Medallion pipeline\Week_02\Day_07\Lab_Files\lake\bronze\orders_raw


## 6. Audit Counts by Source File

In [7]:
by_file = read_parquet(spark, bronze_path).groupBy("_source_file").count().orderBy("_source_file")
by_file.show(truncate=False)
write_csv_dir(by_file, OUTPUT_DIR / "notebook_04_bronze_counts_by_file")
print(f"Bronze rows written: {bronze.count()}")

+--------------------------+-----+
|_source_file              |count|
+--------------------------+-----+
|order_events_batch_1.jsonl|6    |
|order_events_batch_2.jsonl|6    |
+--------------------------+-----+

Bronze rows written: 12


## Clean Shutdown

Stop the SparkSession when you are done with the notebook. The next notebook will create its own session.

In [8]:
# Run this at the end of the notebook, or before restarting/rerunning the lab.
spark.stop()